# TAPF-MIN — User Video Evidence Notebook

**Purpose:** run the TAPF-MIN video pipeline on the user's own video archive and Kaggle video samples while preserving scientific honesty.

This notebook:
- discovers the uploaded ZIP automatically;
- also supports `/kaggle/input/datasets/yasmeens/vedio-samples`;
- audits every readable video;
- samples frames uniformly;
- applies fail-closed face preprocessing where repository code is available;
- computes motion summaries;
- demonstrates minimum-disclosure randomized-response release;
- never fabricates FER accuracy or clinical claims.

> **Important:** until a trained FER checkpoint is supplied, the notebook uses an explicitly labelled demo posterior only to exercise the privacy-release layer.

In [ ]:
# Optional installs for Kaggle/Colab. Skip if already available.
# !pip -q install opencv-python-headless==4.10.0.84 numpy pandas matplotlib

from pathlib import Path
import os, sys, json, math, zipfile, shutil
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

print('Python:', sys.version.split()[0])
print('OpenCV:', cv2.__version__)


## 1. Dataset configuration
The notebook checks both the current uploaded ZIP and the Kaggle dataset path. You can add more paths without changing the rest of the notebook.

In [ ]:
UPLOAD_ZIP_CANDIDATES = [
    Path('/mnt/data/video sample-20260905T125121Z-1-001.zip'),
    Path('/kaggle/input/video-sample/video sample-20260905T125121Z-1-001.zip'),
]
KAGGLE_VIDEO_DIR = Path('/kaggle/input/datasets/yasmeens/vedio-samples')
EXTRACT_DIR = Path('/kaggle/working/tapf_user_videos') if Path('/kaggle/working').exists() else Path('./tapf_user_videos')
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {'.mp4', '.mov', '.avi', '.mkv', '.flv', '.webm', '.m4v'}

zip_path = next((p for p in UPLOAD_ZIP_CANDIDATES if p.exists()), None)
print('Uploaded ZIP:', zip_path if zip_path else 'not found in current runtime')
print('Kaggle video dir exists:', KAGGLE_VIDEO_DIR.exists())
print('Extraction/work dir:', EXTRACT_DIR.resolve())


In [ ]:
# Safe extraction: reject absolute paths and ../ traversal.
def safe_extract_zip(zip_path: Path, dest: Path):
    extracted = []
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for info in zf.infolist():
            name = info.filename.replace('\\', '/')
            parts = Path(name).parts
            if name.startswith('/') or '..' in parts:
                print('SKIP unsafe archive member:', name)
                continue
            out = dest / name
            out.parent.mkdir(parents=True, exist_ok=True)
            if info.is_dir():
                out.mkdir(parents=True, exist_ok=True)
                continue
            with zf.open(info) as src, open(out, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            extracted.append(out)
    return extracted

if zip_path is not None:
    extracted = safe_extract_zip(zip_path, EXTRACT_DIR)
    print(f'Extracted {len(extracted)} files')
else:
    extracted = []


In [ ]:
def discover_videos(*roots):
    found = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        if root.is_file() and root.suffix.lower() in VIDEO_EXTS:
            found.append(root)
        elif root.is_dir():
            found.extend(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in VIDEO_EXTS)
    # stable de-duplication
    unique = []
    seen = set()
    for p in found:
        key = str(p.resolve())
        if key not in seen:
            seen.add(key); unique.append(p)
    return sorted(unique)

video_files = discover_videos(EXTRACT_DIR, KAGGLE_VIDEO_DIR)
print('Discovered videos:', len(video_files))
for p in video_files[:30]:
    print(' -', p)
if not video_files:
    print('No videos found. Check the configured paths above.')


## 2. Video integrity and metadata audit
Unreadable or empty videos are reported explicitly. No silent exclusion.

In [ ]:
def video_metadata(path: Path):
    cap = cv2.VideoCapture(str(path))
    opened = cap.isOpened()
    fps = float(cap.get(cv2.CAP_PROP_FPS)) if opened else 0.0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if opened else 0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) if opened else 0
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) if opened else 0
    duration = n / fps if opened and fps > 0 else np.nan
    ok_first = False
    if opened:
        ok_first, _ = cap.read()
    cap.release()
    return {
        'file': str(path), 'name': path.name, 'opened': bool(opened),
        'first_frame_readable': bool(ok_first), 'frames': n, 'fps': fps,
        'width': w, 'height': h, 'duration_sec': duration,
        'size_mb': path.stat().st_size / 1024**2,
    }

audit_df = pd.DataFrame([video_metadata(p) for p in video_files])
display(audit_df)
if len(audit_df):
    print('Readable:', int((audit_df.opened & audit_df.first_frame_readable).sum()), '/', len(audit_df))


## 3. Uniform frame sampling
This avoids taking only the first seconds of each clip.

In [ ]:
def read_uniform_frames(path: Path, n_frames=8):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return []
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return []
    idxs = np.linspace(0, max(total - 1, 0), n_frames).round().astype(int)
    frames = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if ok and frame is not None:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames

if video_files:
    sample_path = video_files[0]
    frames = read_uniform_frames(sample_path, 8)
    print('Sample:', sample_path.name, '| frames read:', len(frames))
    if frames:
        fig, axes = plt.subplots(2, 4, figsize=(14, 7))
        for ax, frame in zip(axes.flat, frames):
            ax.imshow(frame); ax.axis('off')
        plt.tight_layout(); plt.show()


## 4. Fail-closed face preprocessing
If the repository's `tapf.face_preprocess` module is available, this cell uses it. Otherwise it falls back to a conservative Haar detector that returns an invalid frame rather than a full-scene center crop.

In [ ]:
REPO_ROOTS = [Path.cwd(), Path('/kaggle/working/vedio-firewall'), Path('/content/vedio-firewall')]
for root in REPO_ROOTS:
    if (root / 'tapf').exists() and str(root) not in sys.path:
        sys.path.insert(0, str(root))

try:
    from tapf.face_preprocess import FacePreprocessor
    repo_preprocessor = FacePreprocessor(output_size=112, max_reuse_frames=2)
    HAVE_REPO_PREPROCESS = True
    print('Using repository FacePreprocessor')
except Exception as exc:
    HAVE_REPO_PREPROCESS = False
    print('Repository FacePreprocessor unavailable:', exc)
    cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def conservative_face_crop(rgb, size=112):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    faces = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40,40))
    if len(faces) == 0:
        neutral = np.full((size, size, 3), 127, dtype=np.uint8)
        return neutral, False
    x,y,w,h = max(faces, key=lambda b: b[2]*b[3])
    crop = rgb[y:y+h, x:x+w]
    crop = cv2.resize(crop, (size,size), interpolation=cv2.INTER_AREA)
    return crop, True


In [ ]:
def preprocess_video(path: Path, n_frames=8):
    rgb_frames = read_uniform_frames(path, n_frames)
    if not rgb_frames:
        return {'file': str(path), 'valid_rate': 0.0, 'n_frames': 0, 'crops': [], 'valid': []}
    if HAVE_REPO_PREPROCESS:
        # FacePreprocessor in the repo expects BGR/OpenCV frames.
        bgr = [cv2.cvtColor(f, cv2.COLOR_RGB2BGR) for f in rgb_frames]
        seq = repo_preprocessor.process_sequence(bgr)
        # process_sequence returns crops + quality flags in the hardened branch.
        if isinstance(seq, tuple) and len(seq) >= 4:
            crops, detected, aligned, valid = seq[:4]
            crops_rgb = [cv2.cvtColor(c, cv2.COLOR_BGR2RGB) if c.ndim == 3 else c for c in crops]
            return {'file': str(path), 'valid_rate': float(np.mean(valid)), 'n_frames': len(crops_rgb), 'crops': crops_rgb, 'valid': list(map(bool, valid))}
        raise RuntimeError('Unexpected FacePreprocessor.process_sequence output schema')
    crops, valid = [], []
    for f in rgb_frames:
        c, ok = conservative_face_crop(f)
        crops.append(c); valid.append(ok)
    return {'file': str(path), 'valid_rate': float(np.mean(valid)), 'n_frames': len(crops), 'crops': crops, 'valid': valid}

preprocess_rows = []
preprocess_cache = {}
for p in video_files:
    out = preprocess_video(p, 8)
    preprocess_cache[str(p)] = out
    preprocess_rows.append({'file': str(p), 'name': p.name, 'sampled_frames': out['n_frames'], 'face_valid_rate': out['valid_rate']})
preprocess_df = pd.DataFrame(preprocess_rows)
display(preprocess_df)


## 5. Motion preservation proxy
This is a development diagnostic, not a clinical metric. It summarizes adjacent-frame motion energy inside the preprocessed face sequence.

In [ ]:
def motion_energy(crops, valid):
    if len(crops) < 2:
        return np.nan
    vals = []
    for i in range(1, len(crops)):
        if not (valid[i-1] and valid[i]):
            continue
        a = crops[i-1].astype(np.float32) / 255.0
        b = crops[i].astype(np.float32) / 255.0
        vals.append(float(np.mean(np.abs(b-a))))
    return float(np.mean(vals)) if vals else np.nan

motion_rows = []
for p in video_files:
    d = preprocess_cache[str(p)]
    motion_rows.append({'name': p.name, 'face_valid_rate': d['valid_rate'], 'motion_energy': motion_energy(d['crops'], d['valid'])})
motion_df = pd.DataFrame(motion_rows)
display(motion_df)


## 6. Minimum-disclosure privacy release
The release mechanism below is the real six-class k-ary randomized response. The task posterior is **demo-only** unless you load a trained FER checkpoint.

In [ ]:
EMOTIONS = ('ANG','DIS','FEA','HAP','NEU','SAD')
EMOTION_NAMES = dict(ANG='Anger', DIS='Disgust', FEA='Fear', HAP='Happiness', NEU='Neutral', SAD='Sadness')

try:
    from tapf.formal_privacy import randomized_response
except Exception:
    def randomized_response(value, labels, epsilon, rng=None, accountant=None):
        rng = np.random.default_rng() if rng is None else rng
        labels = tuple(labels)
        k = len(labels)
        p_true = math.exp(epsilon)/(math.exp(epsilon)+k-1)
        probs = np.full(k, (1-p_true)/(k-1))
        probs[labels.index(value)] = p_true
        return {'label': labels[int(rng.choice(k, p=probs))], 'epsilon': epsilon}

DEMO_POSTERIOR = np.array([0.06,0.05,0.04,0.61,0.17,0.07], dtype=float)
DEMO_POSTERIOR /= DEMO_POSTERIOR.sum()
local_label = EMOTIONS[int(np.argmax(DEMO_POSTERIOR))]
epsilon = 1.0
rr = randomized_response(local_label, EMOTIONS, epsilon=epsilon, rng=np.random.default_rng(42), accountant=None)
release_payload = {
    'type': 'categorical_label',
    'value': rr['label'],
    'payload_bits': int(math.ceil(math.log2(len(EMOTIONS)))),
    'raw_video_released': False,
    'task_latent_released': False,
    'epsilon': epsilon,
    'source_status': 'DEMO_POSTERIOR_NOT_BENCHMARK_FER',
}
print('Local demo label:', EMOTION_NAMES[local_label])
print(json.dumps(release_payload, indent=2))


## 7. Export evidence manifest
This exports only metadata and measured development diagnostics. It does not export raw frames or face crops.

In [ ]:
OUT_DIR = Path('/kaggle/working/tapf_results') if Path('/kaggle/working').exists() else Path('./tapf_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)
audit_df.to_csv(OUT_DIR/'user_video_audit.csv', index=False)
preprocess_df.to_csv(OUT_DIR/'user_video_preprocess_quality.csv', index=False)
motion_df.to_csv(OUT_DIR/'user_video_motion_summary.csv', index=False)
manifest = {
    'dataset_type': 'user_supplied_video_samples',
    'video_count': len(video_files),
    'readable_count': int((audit_df.opened & audit_df.first_frame_readable).sum()) if len(audit_df) else 0,
    'raw_video_released': False,
    'task_latent_released': False,
    'fer_benchmark_status': 'NOT_CLAIMED_WITHOUT_TRAINED_CHECKPOINT',
    'privacy_demo': release_payload,
}
with open(OUT_DIR/'tapf_user_video_manifest.json','w') as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))
print('Saved to:', OUT_DIR.resolve())


## 8. Next benchmark stage
For publication-quality evidence, use the same pipeline with:
1. **CREMA-D** — controlled actor-disjoint FER/privacy benchmark;
2. **Aff-Wild2** — stronger in-the-wild FER validation;
3. **BP4D / DISFA** — facial Action Unit and motion-utility preservation;
4. **AVA-ActiveSpeaker** — speaking/communication utility.

The user's own videos are useful for pipeline robustness and qualitative execution tests, but they should not replace actor-disjoint benchmark datasets for scientific performance claims.